# PE toy experiment — run it all on Colab

Everything the harness does, off your laptop. Nothing here needs a GPU;
a plain **CPU runtime** is right. Pick **High-RAM** only for `q0_scale`.

Work is mirrored to Google Drive after every stage, so a dropped runtime
costs you nothing — re-run the cells and each stage resumes from its
`results/*.jsonl` cache.

**Order:** 1 Drive → 2 code → 3 deps → 4 caches → 5 grid → 6 tables →
7 scale-up (optional) → 8 bring results home.


## 1. Mount Drive

This is what makes a disconnect harmless. Everything lands in
`MyDrive/pe_toy/` (~400 MB once the embedding cache is built).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pe_toy

## 2. Get the code

**Option A — git (preferred).** Works once `ConPE/` is committed and pushed:


In [ ]:
%cd /content
!rm -rf PE-exps
!git clone -q https://github.com/KayorKando/PE-exps.git
%cd /content/PE-exps/ConPE/Toy-exp

# ipykernel puts the *startup* dir (/content) on sys.path, not the current one,
# so `%cd` alone leaves `import colab` / `import pe_toy` failing.
import os, sys
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('ROOT', ROOT)
!ls


**Option B — zip upload.** Use this if the code isn't pushed yet. On your
laptop run `bash make_bundle.sh` in `ConPE/Toy-exp/`, then run this cell and
pick `pe_toy_bundle.zip` (64 KB — code only).


In [ ]:
# from google.colab import files
# %cd /content
# up = files.upload()
# !mkdir -p /content/toyexp && unzip -oq pe_toy_bundle.zip -d /content/toyexp
# %cd /content/toyexp
# import os, sys
# ROOT = os.getcwd()
# if ROOT not in sys.path:
#     sys.path.insert(0, ROOT)
# !ls


## 3. Dependencies and machine check


In [ ]:
!pip -q install sentence-transformers
import multiprocessing as mp, numpy, scipy, sklearn
print('numpy',numpy.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__)
print('cores', mp.cpu_count())
!free -g | head -2
# A config A run peaks ~0.5 GB, config B at D=768 ~0.8 GB.
# Colab standard is ~12 GB / 2 cores; JOBS=2 there, 4 on a bigger runtime.
JOBS = min(4, mp.cpu_count())
print('JOBS =', JOBS)

## 4. Restore anything already done, then build the caches

`pull()` copies previous `results/` and `.cache/` back from Drive. On a
first run there is nothing to restore and the cache build takes ~5 min:
24k Yelp sentences → all-mpnet-base-v2 → PCA to 128 and 768 → each seed's
`Z0`.

That `Z0` step is what made stage 5 slow before it was cached — a
2-component full-covariance GMM on a 20000×768 matrix, ~60 s per (D, seed),
which used to be refitted inside **every** run. Warming it serially here
also stops parallel workers racing on the same fit.


In [ ]:
from colab.sync import pull, push, status
pull()

import time
from pe_toy.data import fetch_corpus, embed_corpus, pca_pool, make_dataset
sents = fetch_corpus('yelp', 24000); print('sentences', len(sents))
print('embeddings', embed_corpus(sents).shape)
for D in (128, 768):
    pca_pool('yelp', D); print('pca pool', D, 'ready', flush=True)
for D in (128, 768):
    for s in range(5):
        t = time.time(); make_dataset('B', D=D, N=20000, n_syn=500, n_ref=2000, seed=s)
        print(f'  Z0 D={D} seed={s}  {time.time()-t:.0f}s', flush=True)
push()

## 5. The v0.2 grid

Stage-gated and strictly sequential. Stage 0 is the Q0 kill (one round, no
loop); stage 1 is the blocking health gate — nothing later is read from a
cell that fails it.

**Pick a mode first.** The repo ships with `results/` already populated from
the run on the laptop, so by default every stage reports *all cached* and
does nothing. That is what you want if you are here for the scale-up sweeps
in step 7; it is not what you want if you are reproducing from scratch.

- `MODE = 'resume'` — keep the shipped results, only fill gaps. Fast.
- `MODE = 'fresh'`  — wipe `results/` and rerun everything (~40 min).

Either way each stage pushes to Drive when it finishes, so if the runtime
drops mid-stage you just re-run the cell and it picks up where it stopped.


In [ ]:
MODE = 'resume'      # 'resume' keeps the shipped results | 'fresh' rewrites them

import shutil, pathlib
if MODE == 'fresh':
    shutil.rmtree('results', ignore_errors=True)
    pathlib.Path('results').mkdir()
    print('results/ cleared -- full rerun')
else:
    n = sum(1 for f in pathlib.Path('results').glob('*.jsonl') for _ in f.open())
    print(f'resuming; {n} runs already on disk')

In [ ]:
import subprocess
from colab.sync import push

for stage_args in (['0'], ['1'], ['2','3','4'], ['5']):
    cmd = ['python','-m','pe_toy.run_grid', *stage_args, '--jobs', str(JOBS)]
    print('>>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    push()

## 6. Tables and figures


In [ ]:
!python -m pe_toy.analyze | tail -80
!python -m pe_toy.figures
from colab.sync import push; push()

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')):
    print(f); display(Image(f, width=900))

## 7. Scale-up sweeps (optional)

The open questions run #2 could not afford. Run them one at a time; each is
resumable and pushes to Drive on completion.

| sweep | question | cost |
|---|---|---|
| `nsyn_health` | Can a bigger `n_syn` buy health at D ≥ 32? **Blocking for v0.3** — without a healthy non-trivial cell, Q1 cannot be asked at all | ~2.5 h |
| `configA_768` | The §3.4 breakage check on config A, never run in run #2 | ~20 min |
| `q0_scale` | Is the D=128 direction plateau (cos ≈ 0.09, flat from N=2k→100k) still there at N=1e6? | ~25 min, **High-RAM** |

For the 2.5 h one, keep the tab open — Colab reclaims idle runtimes.


In [ ]:
!python -m colab.scale_up nsyn_health --jobs {JOBS}
from colab.sync import push; push()

In [ ]:
!python -m colab.scale_up configA_768 --jobs {JOBS}
# !python -m colab.scale_up q0_scale --jobs 1     # High-RAM runtime only
!python -m colab.scale_up report
from colab.sync import push; push()

## 8. Bring the results home

They are already in `MyDrive/pe_toy/results` — this is just a direct download.
On the laptop, drop `results/` and `figures/` into `ConPE/Toy-exp/` and run
`python3 -m pe_toy.analyze` to regenerate the tables locally.


In [ ]:
from colab.sync import status; status()
!zip -qr results_out.zip results figures
from google.colab import files; files.download('results_out.zip')